In [1]:
import sys
sys.path.append("..")

import pandas as pd

short_ic_ranking = pd.read_excel("../exports/RANKING_short_iron_condor.xlsx")

# Filter out gap-risk-flagged AND volatility-persistence-flagged stocks
clean_candidates = short_ic_ranking[
    (short_ic_ranking["gap_risk_flag"].isna() | (short_ic_ranking["gap_risk_flag"] == "")) &
    (short_ic_ranking["volatility_persistence_flag"].isna() | (short_ic_ranking["volatility_persistence_flag"] == ""))
]
top_25 = clean_candidates.head(25)

must_include = short_ic_ranking[short_ic_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[
    (must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")) &
    (must_include["volatility_persistence_flag"].isna() | (must_include["volatility_persistence_flag"] == ""))
]

final_candidates = pd.concat([top_25, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,short_ic_score,gap_risk_flag,Total flips (closed),Median monthly range_pct,recency_weighted_range_pct,Largest monthly range_pct,Median max_adverse_pct,recency_weighted_adverse_pct,Worst max_adverse_pct (4yr),Median duration (days),Worst single gap (4yr),last_month_range_pct,volatility_persistence_flag
0,NIFTY,98.2,NaN,79,5.36,5.04,12.39,1.02,0.91,5.95,15.2,2.32,6.54,NaN
1,FINNIFTY,97.4,NaN,83,6.15,5.48,15.57,1.27,1.11,7.04,14.2,2.35,7.01,NaN
2,SENSEX,97.1,NaN,79,5.50,5.50,12.32,1.06,1.28,5.55,16.0,2.31,6.71,NaN
3,MIDCPNIFTY,96.1,NaN,81,7.06,7.14,16.90,1.86,1.50,7.40,14.0,2.84,9.02,NaN
4,NIFTYNXT50,93.5,NaN,85,7.24,6.88,18.74,1.21,1.06,9.24,15.2,2.80,5.47,NaN
5,BANKNIFTY,88.8,NaN,73,6.20,5.17,17.01,1.53,1.82,7.15,20.2,3.73,6.92,NaN
6,SUNPHARMA,86.8,NaN,53,8.36,7.65,16.84,2.57,2.38,7.31,20.0,3.53,7.58,NaN
7,ULTRACEMCO,85.3,NaN,70,8.45,7.96,22.72,2.43,2.54,7.21,19.0,3.61,7.96,NaN
8,HDFCBANK,84.9,NaN,67,8.09,9.03,26.94,2.19,2.54,7.56,17.0,3.33,9.56,NaN
9,GRASIM,84.4,NaN,67,9.02,8.71,21.67,2.42,2.63,6.95,19.0,3.44,8.71,NaN


In [2]:
# Incremental refresh — same pattern as 06_daily_screener.ipynb/02_position_monitor.ipynb.
# Previously this did a full fetch(from_date=2022-01-01) for every one of the ~27
# symbols on every run - not destructive (the from_date was always the same, so it
# was never silently shrinking history), just wasteful: re-pulling ~4 years of 4H
# data for a shortlist check that only needs to catch up on what changed since the
# last save. process_stock_incremental() fetches just the recent catch-up window
# and merges onto existing saved history instead.
#
# index_symbols also only had NIFTY/BANKNIFTY before this fix - a real gap, since
# FINNIFTY/SENSEX/MIDCPNIFTY/NIFTYNXT50 are 4 of the top 6 short_ic_score names
# (indices score best here: low range, tight adverse) and were silently skipped
# with "no token found" every run. Added their correct tradingsymbol/exchange pairs.

from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {
    "NIFTY": ("NIFTY 50", "NSE"),
    "BANKNIFTY": ("NIFTY BANK", "NSE"),
    "FINNIFTY": ("NIFTY FIN SERVICE", "NSE"),
    "SENSEX": ("SENSEX", "BSE"),
    "MIDCPNIFTY": ("NIFTY MID SELECT", "NSE"),
    "NIFTYNXT50": ("NIFTY NEXT 50", "NSE"),
}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_stock_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-06-29 to 2026-10-27...
Saved 2376 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/NIFTY_full_report.xlsx
[1/25] NIFTY — refreshed
Fetching 2026-06-29 to 2026-10-27...
Saved 2376 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/FINNIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/FINNIFTY_full_report.xlsx
[2/25] FINNIFTY — refreshed
Fetching 2026-06-29 to 2026-10-27...
Saved 2377 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SENSEX_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/SENSEX_full_report.xlsx
[3/25] SENSEX — refreshed
Fetching 2026-06-29 to 2026-10-27...
Saved 2348 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/MIDCPNIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/MIDCPNIFTY_full_report.xlsx
[4/25] MIDCPNIFTY — refreshed
Fetching 2026-06-

In [3]:
from src.monthly_range_screener import screen_monthly_range

live_check = screen_monthly_range(symbols_to_refresh)
live_check.sort_values("current_month_range_pct", ascending=False)

,symbol,days_into_month,current_month_range_pct,median_monthly_range,75th_pct_monthly_range,90th_pct_monthly_range,already_past_median,already_past_75th,already_past_90th
15,TCS,27,28.25,9.71,11.52,14.41,True,True,True
17,COLPAL,27,16.38,9.32,12.16,18.63,True,True,False
23,CONCOR,27,11.22,12.00,17.14,21.09,False,False,False
18,SHREECEM,27,10.49,10.36,13.80,16.89,True,False,False
10,DABUR,27,10.23,8.74,10.66,14.92,True,False,False
11,BAJAJ-AUTO,27,9.99,10.80,14.08,16.73,False,False,False
20,COALINDIA,27,9.56,12.08,14.59,16.67,False,False,False
12,KOTAKBANK,27,9.24,8.74,11.15,13.65,True,False,False
24,BOSCHLTD,27,9.23,10.32,12.94,19.98,False,False,False
19,EICHERMOT,27,8.96,11.68,14.56,17.96,False,False,False
